# Fundamentos de Ingeniería de Datos
## Extracción: de la capa RAW a la capa EXTRACTED

```
INGESTA (ya hecho)                          EXTRACCIÓN (hoy)
Kaggle → CSV → SQLite: raw_df1    →    SQLite: extracted_df1
                                       + control de cambios (cdc_control.json)
                                       + log de auditoría   (extraccion_log.jsonl)
```

**Tema 2.1 — Extracción de datos:** formatos y control de cambios.

### ¿Qué es la capa EXTRACTED?

Son **los mismos datos de `raw_df1`**, sin limpiar todavía, pero con **4 columnas de metadatos** que dicen cuándo, de dónde y en qué lote llegó cada fila.

| Capa | Tabla | Qué contiene |
|---|---|---|
| RAW | `raw_df1` | Datos originales, tal como llegaron (no se modifica) |
| **EXTRACTED** | **`extracted_df1`** | **Los mismos datos + metadatos de extracción** |
| TRANSFORMED | `transformed_df1` | Datos limpios (siguiente notebook) |

> **Regla de oro:** cada capa lee solo de la anterior. `raw_df1` se lee en **modo solo lectura**.

### Configuración del entorno

In [101]:
# # ── Montar Google Drive ────────────────────────────────────────────────────

# from google.colab import drive
# drive.mount('/content/drive')

In [102]:
# ── Importar utils.py ──────────────────────────────────────────────────────

import sys
sys.path.append('/Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Utilidades')

from utils import PROYECTO_BASE, DB_PATH, inspeccionar_dataset, listar_tablas

print()
print('utils.py importado correctamente')
print(f'Base de datos: {DB_PATH}')
print()


utils.py importado correctamente
Base de datos: /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Ingesta_de_Datos/Base_de_datos/base_datos.db



In [103]:
# ── Librerías, rutas y nombres de tablas ───────────────────────────────────

import os
import json
import hashlib
import sqlite3
import pandas as pd
from datetime import datetime




# ----------- opcional no afecta si deciden colocar el nombre la tabla -----------
TABLA_RAW       = 'raw_df1'          # origen
TABLA_EXTRACTED = 'extracted_df1'    # destino




# ----------- Carpetas de esta capa -----------

EXTRACCION_PATH = f'{PROYECTO_BASE}/Extraccion_de_Datos'
LOGS_PATH       = f'{EXTRACCION_PATH}/Logs'
EXPORTS_PATH    = f'{EXTRACCION_PATH}/Exportaciones'

# ----------- Archivos -----------

CDC_PATH = f'{LOGS_PATH}/cdc_control.json'        # estado del control de cambios
LOG_PATH = f'{LOGS_PATH}/extraccion_log.jsonl'    # historial de operaciones


# -----------  Si no existen, crearlos -----------
os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(EXPORTS_PATH, exist_ok=True)




print()
print(f'Origen  : {TABLA_RAW}')
print(f'Destino : {TABLA_EXTRACTED}')
print(f'Logs    : {LOGS_PATH}')
print()


Origen  : raw_df1
Destino : extracted_df1
Logs    : /Users/artdeco/Documents/CUGDL/Ingenieria_de_datos/Extraccion_de_Datos/Logs



---
## PASO 1 — Revisar el origen: `raw_df1`

Antes de extraer, verificamos que el origen existe y tiene lo que esperamos. Lo abrimos en **modo solo lectura**, como en la clase de SQL.

In [104]:
# ── ¿Qué tablas hay en el .db? ─────────────────────────────────────────────

listar_tablas(DB_PATH);



  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  extracted_df1                       extracted_        16,598     15
  raw_df1                             raw_              16,598     11
  raw_sp2                             raw_                 500     12
  raw_sp3                             raw_                 646     17


In [105]:
# ── Eliminar una tabla ─────────────────────────────────────────────────────

TABLA_A_BORRAR = 'raw_sp1'      # cambia el nombre de la tabla

conn = sqlite3.connect(DB_PATH)       # conexión normal (NO en modo solo lectura)
conn.execute(f'DROP TABLE IF EXISTS {TABLA_A_BORRAR}')
conn.commit()
conn.close()

print(f"Tabla '{TABLA_A_BORRAR}' eliminada")

listar_tablas(DB_PATH);



Tabla 'raw_sp1' eliminada

  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  extracted_df1                       extracted_        16,598     15
  raw_df1                             raw_              16,598     11
  raw_sp2                             raw_                 500     12
  raw_sp3                             raw_                 646     17


In [106]:
# ── Leer raw_df1 en modo solo lectura ──────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

df_raw = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)
conn.close()

inspeccionar_dataset(df_raw, nombre=TABLA_RAW)


  DATASET: raw_df1
  Filas    : 16,598
  Columnas : 11

  Tipos de datos y valores nulos:
    Rank                         int64        nulos: 0 (0.0%)
    Name                         str          nulos: 0 (0.0%)
    Platform                     str          nulos: 0 (0.0%)
    Year                         float64      nulos: 271 (1.6%)
    Genre                        str          nulos: 0 (0.0%)
    Publisher                    str          nulos: 58 (0.3%)
    NA_Sales                     float64      nulos: 0 (0.0%)
    EU_Sales                     float64      nulos: 0 (0.0%)
    JP_Sales                     float64      nulos: 0 (0.0%)
    Other_Sales                  float64      nulos: 0 (0.0%)
    Global_Sales                 float64      nulos: 0 (0.0%)

  Primeras 3 filas:


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82


---
## PASO 2 — Metadatos de extracción

A cada fila le agregamos 4 columnas. Empiezan con guion bajo `_` para distinguirlas de las columnas originales.

| Columna | Contenido | ¿Para qué sirve? |
|---|---|---|
| `_batch_id` | Número de lote: 1 la primera vez, 2 la siguiente… | Saber en qué ejecución llegó cada fila |
| `_extracted_at` | Fecha y hora de la extracción | Medir la **frescura** de los datos |
| `_source_table` | Tabla de origen (`raw_df1`) | **Lineage**: de dónde viene cada fila |
| `_row_hash` | "Huella digital" del contenido de la fila | **Control de cambios** (Paso 3) |

### La huella (`_row_hash`)

Una **función hash** convierte un texto en un código corto. Si el texto cambia, aunque sea un carácter, el código cambia por completo. Juntamos los valores de la fila en un texto y le calculamos su hash: **misma fila → misma huella; fila modificada → huella nueva**.

In [107]:
# ── ¿Cómo funciona un hash? ────────────────────────────────────────────────
print()

for texto in ['Wii Sports|Wii|2006.0',
              'Wii Sports|Wii|2007.0',
              'Wii Sports|Wii|2006.0']:

    huella = hashlib.sha256(texto.encode('utf-8')).hexdigest()[:16]
    print(f'{texto:<25} →  {huella}')

print()


Wii Sports|Wii|2006.0     →  a2cb45e64746c2e9
Wii Sports|Wii|2007.0     →  197bc619b99265e0
Wii Sports|Wii|2006.0     →  a2cb45e64746c2e9



In [108]:
# ── Función para calcular la huella de cada fila ───────────────────────────

def calcular_huella(df):

    '''Regresa la huella
    (16 caracteres) de cada
    fila del DataFrame.'''


    texto_fila = df.map(str).agg('|'.join, axis=1)              # 'v1|v2|v3|...'
    return texto_fila.map(lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16])



In [109]:
# ── Prueba con las primeras 3 filas ───────────────────────────

prueba = df_raw.head(3).copy()
prueba['_row_hash'] = calcular_huella(df_raw.head(3))
prueba


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_row_hash
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,4afd4e8c1f96d38e
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,3ee71d62807f116a
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,688fb7c1f6ec1137


---
## PASO 3 — Control de cambios (CDC)

Este pipeline no se ejecuta una sola vez: se vuelve a ejecutar cada vez que hay una nueva ingesta (según la **frescura** de tu miniespecificación). El **control de cambios** evita copiar todo otra vez:

| Tipo de carga | Qué hace |
|---|---|
| **Completa** (*full load*) | Copia toda la tabla. Ocurre la **primera vez**. |
| **Incremental** | Copia **solo las filas nuevas o modificadas**. Ocurre en las siguientes ejecuciones. |

**¿Cómo sabemos qué es nuevo?** Comparamos la huella de cada fila de `raw_df1` con las huellas que **ya están** en `extracted_df1`. Solo pasan las que no estaban.

> Usamos huellas porque `raw_df1` no tiene una columna de fecha de registro. Si la tuviera, podríamos usar `WHERE fecha > última_fecha_extraída` (estrategia de **marca de agua**).

Guardamos el estado en `cdc_control.json` (último lote, fecha, filas) y cada operación en el log.

In [110]:
# ── Funciones de control de cambios y log ──────────────────────────────────

# ── Leer cdc ──────────────────────────────────
def leer_cdc():

    '''Lee el estado del control de cambios (cdc_control.json)
       vacío si es la primera vez '''


    if not os.path.exists(CDC_PATH):
        return {}
    with open(CDC_PATH, encoding='utf-8') as f:
        return json.load(f)


# ── Guardar cdc ──────────────────────────────────

def guardar_cdc(estado):
    with open(CDC_PATH, 'w', encoding='utf-8') as f:
        json.dump(estado, f, indent=2, ensure_ascii=False)



# ── Escribe cdc ──────────────────────────────────

def escribir_log(filas_leidas, filas_nuevas, tipo, estado='exitoso', error=None):

    '''Agrega UNA línea al log (formato JSON Lines: un JSON por línea).'''

    evento = {
        'timestamp'    : datetime.now().isoformat(timespec='seconds'),
        'tabla_origen' : TABLA_RAW,
        'tabla_destino': TABLA_EXTRACTED,
        'filas_leidas' : filas_leidas,
        'filas_nuevas' : filas_nuevas,
        'tipo'         : tipo,
        'estado'       : estado,
        'error'        : error,
    }

    with open(LOG_PATH, 'a', encoding='utf-8') as f:     # 'a' = agregar al final
        f.write(json.dumps(evento, ensure_ascii=False) + '\n')






# ── Historial ──────────────────────────────────

def ver_log():
    if not os.path.exists(LOG_PATH):
        print('Todavía no hay operaciones registradas.')
        return pd.DataFrame()
    return pd.read_json(LOG_PATH, lines=True)




print()
print('Funciones listas')
print()


Funciones listas



In [111]:
print(leer_cdc())

{'ultimo_batch': 2, 'tabla_origen': 'raw_df1', 'tabla_destino': 'extracted_df1', 'ultima_extraccion': '2026-10-29T08:49:43', 'filas_leidas': 16598, 'filas_nuevas': 0, 'filas_acumuladas': 16598, 'tipo': 'incremental'}


In [112]:
listar_tablas(DB_PATH)


  Tablas en: base_datos.db

  TABLA                               CAPA               FILAS   COLS
  ----------------------------------- --------------- -------- ------
  extracted_df1                       extracted_        16,598     15
  raw_df1                             raw_              16,598     11
  raw_sp2                             raw_                 500     12
  raw_sp3                             raw_                 646     17


['extracted_df1', 'raw_df1', 'raw_sp2', 'raw_sp3']

In [113]:
# ── La función de extracción ───────────────────────────────────────────────

def extraer():

    '''Extrae raw_df1 → extracted_df1, agregando solo las filas nuevas.'''


    try:

        # 1. Leer el origen (solo lectura)
        conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
        df = pd.read_sql_query(f'SELECT * FROM {TABLA_RAW}', conn)



        # 2. Huellas que ya están en EXTRACTED (si la tabla existe)

        existe = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master WHERE type = 'table' AND name = ?",
            (TABLA_EXTRACTED,)
        ).fetchone()[0] > 0


        if existe:
            previas = set(pd.read_sql_query(f'SELECT _row_hash FROM {TABLA_EXTRACTED}', conn)['_row_hash'])
        else:
            previas = set()
        conn.close()



        # 3. Calcular la huella de cada fila y quedarse solo con las nuevas

        df['_row_hash'] = calcular_huella(df)
        nuevas = df[~df['_row_hash'].isin(previas)].copy()

        tipo = 'full_load' if not previas else 'incremental'
        cdc = leer_cdc()




        # 4. Si hay filas nuevas: metadatos + AGREGAR (append) a extracted_df1

        if len(nuevas) > 0:

            batch_id = cdc.get('ultimo_batch', 0) + 1

            nuevas['_batch_id']     = batch_id
            nuevas['_extracted_at'] = datetime.now().isoformat(timespec='seconds')
            nuevas['_source_table'] = TABLA_RAW


            # Orden: columnas originales y al final los metadatos
            columnas = [c for c in nuevas.columns if not c.startswith('_')]
            nuevas = nuevas[columnas + ['_batch_id', '_extracted_at', '_source_table', '_row_hash']]

            conn = sqlite3.connect(DB_PATH)                      # lectura y escritura
            nuevas.to_sql(TABLA_EXTRACTED, conn, if_exists='append', index=False)
            conn.commit()
            conn.close()

            cdc['ultimo_batch'] = batch_id



        # 5. Actualizar control de cambios y log

        cdc.update({

            'tabla_origen'     : TABLA_RAW,
            'tabla_destino'    : TABLA_EXTRACTED,
            'ultima_extraccion': datetime.now().isoformat(timespec='seconds'),
            'filas_leidas'     : len(df),
            'filas_nuevas'     : len(nuevas),
            'filas_acumuladas' : len(previas) + len(nuevas),
            'tipo'             : tipo,
        })


        guardar_cdc(cdc)



        escribir_log(len(df), len(nuevas), tipo)

        print()
        print(f'{TABLA_RAW} → {TABLA_EXTRACTED}  ({tipo})')
        print(f'  Filas leídas : {len(df):,}')
        print(f'  Filas nuevas : {len(nuevas):,}')
        print()
        return nuevas



    except Exception as e:
        escribir_log(0, 0, 'error', estado='fallido', error=str(e))
        raise          # muestra el error en pantalla


print()
print('extraer() definida')
print()


extraer() definida



> **¿Por qué `if_exists='append'` y no `'replace'`?** `replace` borraría la tabla y la volvería a crear: perderíamos los lotes anteriores. Con `append` las filas nuevas se **agregan al final** y, gracias al control de cambios, nunca se duplican.

---
## PASO 4 — Ejecutar la extracción

**Ejecuta esta celda dos veces** y compara:

- **1ª vez:** carga completa (`full_load`): todas las filas son nuevas.
- **2ª vez:** carga incremental (`incremental`): `raw_df1` no cambió, así que **0 filas nuevas**.

In [114]:
# ── Ejecutar ───────────────────────────────────────────────────────────────

df_nuevas = extraer()


raw_df1 → extracted_df1  (incremental)
  Filas leídas : 16,598
  Filas nuevas : 0



In [115]:
# ── Ver la tabla EXTRACTED ─────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED} LIMIT 5', conn))

conn.close()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_batch_id,_extracted_at,_source_table,_row_hash
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,2,2026-10-21T18:04:50,raw_df1,4afd4e8c1f96d38e
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,2,2026-10-21T18:04:50,raw_df1,3ee71d62807f116a
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,2,2026-10-21T18:04:50,raw_df1,688fb7c1f6ec1137
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,2,2026-10-21T18:04:50,raw_df1,a60698219af7ed96
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,2,2026-10-21T18:04:50,raw_df1,2644afa96579b93b


In [116]:
# ── Estado del control de cambios ──────────────────────────────────────────

print(json.dumps(leer_cdc(), indent=2, ensure_ascii=False))



{
  "ultimo_batch": 2,
  "tabla_origen": "raw_df1",
  "tabla_destino": "extracted_df1",
  "ultima_extraccion": "2026-10-29T08:49:55",
  "filas_leidas": 16598,
  "filas_nuevas": 0,
  "filas_acumuladas": 16598,
  "tipo": "incremental"
}


In [117]:
# ── Log de operaciones ─────────────────────────────────────────────────────

ver_log()



,timestamp,tabla_origen,tabla_destino,filas_leidas,filas_nuevas,tipo,estado,error
0,2026-10-21 17:33:48,raw_df1,extracted_df1,16598,16598,full_load,exitoso,NaN
1,2026-10-21 18:04:50,raw_df1,extracted_df1,16598,16598,full_load,exitoso,NaN
2,2026-10-21 18:08:09,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
3,2026-10-21 18:20:34,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
4,2026-10-21 18:48:25,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
5,2026-10-29 08:44:11,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
6,2026-10-29 08:45:07,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
7,2026-10-29 08:45:41,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
8,2026-10-29 08:49:43,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN
9,2026-10-29 08:49:55,raw_df1,extracted_df1,16598,0,incremental,exitoso,NaN


In [118]:
# ── Así se ve el archivo del log por dentro ────

with open(LOG_PATH, encoding='utf-8') as f:
    print(f.read())


{"timestamp": "2026-10-21T17:33:48", "tabla_origen": "raw_df1", "tabla_destino": "extracted_df1", "filas_leidas": 16598, "filas_nuevas": 16598, "tipo": "full_load", "estado": "exitoso", "error": null}
{"timestamp": "2026-10-21T18:04:50", "tabla_origen": "raw_df1", "tabla_destino": "extracted_df1", "filas_leidas": 16598, "filas_nuevas": 16598, "tipo": "full_load", "estado": "exitoso", "error": null}
{"timestamp": "2026-10-21T18:08:09", "tabla_origen": "raw_df1", "tabla_destino": "extracted_df1", "filas_leidas": 16598, "filas_nuevas": 0, "tipo": "incremental", "estado": "exitoso", "error": null}
{"timestamp": "2026-10-21T18:20:34", "tabla_origen": "raw_df1", "tabla_destino": "extracted_df1", "filas_leidas": 16598, "filas_nuevas": 0, "tipo": "incremental", "estado": "exitoso", "error": null}
{"timestamp": "2026-10-21T18:48:25", "tabla_origen": "raw_df1", "tabla_destino": "extracted_df1", "filas_leidas": 16598, "filas_nuevas": 0, "tipo": "incremental", "estado": "exitoso", "error": null}
{

### (Opcional) Empezar de cero

Borra `extracted_df1` y el control de cambios para volver a hacer la carga completa. **No toca `raw_df1`.** Quita los `#` para ejecutarla.

In [119]:
# conn = sqlite3.connect(DB_PATH)
# conn.execute(f'DROP TABLE IF EXISTS {TABLA_EXTRACTED}')
# conn.commit()
# conn.close()
# if os.path.exists(CDC_PATH):
#     os.remove(CDC_PATH)
# print('extracted_df1 y control de cambios reiniciados')

---
## PASO 5 — Verificar con SQL

Una extracción no está terminada hasta que se **verifica**.

In [120]:
# ── ¿Coinciden RAW y EXTRACTED? ────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT (SELECT COUNT(*) FROM {TABLA_RAW})                       AS filas_raw,
           (SELECT COUNT(*) FROM {TABLA_EXTRACTED})                 AS filas_extracted,
           (SELECT COUNT(DISTINCT _row_hash) FROM {TABLA_EXTRACTED}) AS huellas_distintas
''', conn))

conn.close()

,filas_raw,filas_extracted,huellas_distintas
0,16598,16598,16598


- Si `filas_raw` = `filas_extracted`: la extracción está completa.
- Si `huellas_distintas` es **menor** que `filas_raw`: hay **filas exactamente duplicadas** en `raw_df1`. No es un error de la extracción; se anota para limpiarlo en TRANSFORMED.

In [121]:
# ── ¿Cuántas filas llegaron en cada lote? ──────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT _batch_id, _extracted_at, _source_table, COUNT(*) AS filas
    FROM {TABLA_EXTRACTED}
    GROUP BY _batch_id, _extracted_at, _source_table
    ORDER BY _batch_id
''', conn))

conn.close()

,_batch_id,_extracted_at,_source_table,filas
0,2,2026-10-21T18:04:50,raw_df1,16598


In [122]:
# ── ¿Los metadatos están completos? (no debe haber nulos) ──────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)

display(pd.read_sql_query(f'''
    SELECT COUNT(*)                                                 AS filas,
           SUM(CASE WHEN _batch_id     IS NULL THEN 1 ELSE 0 END)   AS nulos_batch,
           SUM(CASE WHEN _extracted_at IS NULL THEN 1 ELSE 0 END)   AS nulos_fecha,
           SUM(CASE WHEN _row_hash     IS NULL THEN 1 ELSE 0 END)   AS nulos_huella
    FROM {TABLA_EXTRACTED}
''', conn))

conn.close()

,filas,nulos_batch,nulos_fecha,nulos_huella
0,16598,0,0,0


---
## PASO 6 — Formatos: CSV vs. JSON vs. Parquet

Muchas fuentes entregan **archivos** en lugar de tablas SQL. Exportamos `extracted_df1` en los tres formatos más comunes y comparamos.

| Formato | Cómo guarda los datos | Legible por humanos |
|---|---|---|
| **CSV** | Texto, una fila por línea | Sí |
| **JSON** | Texto con pares `"clave": valor` | Sí |
| **Parquet** | Binario, **por columnas**, comprimido y con tipos de dato | No |

In [123]:
# ── Leer extracted_df1 ─────────────────────────────────────────────────────

conn = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
df_ext = pd.read_sql_query(f'SELECT * FROM {TABLA_EXTRACTED}', conn)
conn.close()


In [124]:
# SQLite no tiene tipo fecha: _extracted_at llegó como texto. La convertimos a fecha real.
print()
df_ext['_extracted_at'] = pd.to_datetime(df_ext['_extracted_at'])

df_ext.dtypes.to_frame('tipo').T


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,_batch_id,_extracted_at,_source_table,_row_hash
tipo,int64,str,str,float64,str,str,float64,float64,float64,float64,float64,int64,datetime64[us],str,str


In [125]:
# ── Exportar en los tres formatos ──────────────────────────────────────────

ruta_csv     = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.csv'
ruta_json    = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.json'
ruta_parquet = f'{EXPORTS_PATH}/{TABLA_EXTRACTED}.parquet'


df_ext.to_csv(ruta_csv, index=False)
df_ext.to_json(ruta_json, orient='records', date_format='iso', force_ascii=False)
df_ext.to_parquet(ruta_parquet, index=False)



pd.DataFrame({
    'formato'  : ['CSV', 'JSON', 'Parquet'],
    'tamaño_MB': [round(os.path.getsize(r) / 1_048_576, 2) for r in (ruta_csv, ruta_json, ruta_parquet)],
}).set_index('formato')



ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [ ]:
# ── Leerlos de vuelta: ¿qué tipos de dato se conservan? ────────────────────

desde_csv     = pd.read_csv(ruta_csv)
desde_json    = pd.read_json(ruta_json, orient='records')
desde_parquet = pd.read_parquet(ruta_parquet)

columnas = ['Year', 'Global_Sales', '_batch_id', '_extracted_at']

pd.DataFrame({
    'original': df_ext[columnas].dtypes.astype(str),
    'CSV'     : desde_csv[columnas].dtypes.astype(str),
    'JSON'    : desde_json[columnas].dtypes.astype(str),
    'Parquet' : desde_parquet[columnas].dtypes.astype(str),
})

,original,CSV,JSON,Parquet
Year,float64,float64,float64,float64
Global_Sales,float64,float64,float64,float64
_batch_id,int64,int64,int64,int64
_extracted_at,datetime64[ns],object,datetime64[ns],datetime64[ns]


**Lectura del resultado:**

- **Tamaño:** Parquet es el más pequeño (comprime cada columna). JSON es el más grande porque repite el nombre de cada columna en cada registro.
- **Tipos:** **CSV** pierde la fecha (`_extracted_at` regresa como texto): solo guarda texto y pandas adivina los tipos al leer. **JSON** no tiene tipo fecha; pandas la reconoce porque la columna termina en `_at`, pero es una suposición. **Parquet** guarda el tipo de cada columna dentro del archivo.

| Si necesitas… | Usa |
|---|---|
| Compartir datos para abrir en Excel | **CSV** |
| Intercambiar datos con una API | **JSON** |
| Guardar tablas grandes para análisis | **Parquet** |

---
### Resumen

| Paso | Qué hicimos |
|---|---|
| 1 | Leímos `raw_df1` en **solo lectura** |
| 2 | Agregamos **metadatos**: `_batch_id`, `_extracted_at`, `_source_table`, `_row_hash` |
| 3 | **Control de cambios** con huellas: solo se agregan filas nuevas (`append`) |
| 4 | Registramos cada ejecución en `cdc_control.json` y en el **log** |
| 5 | **Verificamos** con SQL: conteos, lotes y metadatos completos |
| 6 | Comparamos **CSV, JSON y Parquet** |

**Siguiente notebook:** TRANSFORMED, que leerá **`extracted_df1`** (nunca `raw_df1`) para limpiar los datos.